# fase_3 - script_afrida Migration

This notebook handles migration of database from old DB to new DB for fase 3.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [45]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [46]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


## 2. Surat Keluar dan Verifikasi Surat Keluar

In [47]:
# Ambil data suratkeluar
df_sk_raw = pd.read_sql("SELECT * FROM suratkeluar", db_old)
print("=== SURAT KELUAR (lama) ===")
print("Jumlah:", len(df_sk_raw))
display(df_sk_raw.head(5))
print("\nKolom:", df_sk_raw.columns.tolist())
print("\nTipe data:")
print(df_sk_raw.dtypes)
print("\nMissing values:")
print(df_sk_raw.isnull().sum())

# Cek nilai unik kolom status (untuk enum)
print("\nNilai unik 'status':", df_sk_raw['status'].unique())

# Cek apakah idsurat unik?
print("\nApakah idsurat unik?", df_sk_raw['idsurat'].is_unique)

=== SURAT KELUAR (lama) ===
Jumlah: 213


,idsurat,keterangan,link,idusers,status,created_at,nosurat,catatan
0,24,Surat Permohonan Uji Kompetensi dan Penggunaan...,https://docs.google.com/document/d/1xyhc4T-FlR...,U00011,Direvisi,2023-09-11 17:57:49,090A/LEAP/IX/2023- 090B/LEAP/IX/2023,Lengkapi Tanggal Pelaksanaan dengan tanggal ya...
1,26,Beasiswa Siswa GE,https://docs.google.com/document/d/1jMTK_3b57e...,U00011,Disetujui,2023-10-25 16:57:38,101/LEAP/BD/X/2023,
2,27,PERJANJIAN KERJASAMA / MEMORANDUM OF UNDERSTAN...,https://docs.google.com/document/d/1Edeb7hWaBq...,U00011,Disetujui,2023-10-26 17:37:59,102/LEAP/BD/X/2023,
3,28,Sertifikat / Sertifikat Kelas APK Private / 1 ...,https://drive.google.com/drive/folders/1JMrPJF...,U00026,Disetujui,2023-11-10 16:10:04,Sertif 002a/APEX/XI/2324/02 (Page 1) dan 002b/...,
4,29,Penawaran Pelatihan Business English ke PT. La...,https://docs.google.com/document/d/1wBK62noEJw...,U00011,Disetujui,2023-11-13 15:41:58,105/LEAP/BD/XI/2023,



Kolom: ['idsurat', 'keterangan', 'link', 'idusers', 'status', 'created_at', 'nosurat', 'catatan']

Tipe data:
idsurat                int64
keterangan            object
link                  object
idusers               object
status                object
created_at    datetime64[ns]
nosurat               object
catatan               object
dtype: object

Missing values:
idsurat       0
keterangan    0
link          0
idusers       0
status        0
created_at    0
nosurat       0
catatan       0
dtype: int64

Nilai unik 'status': ['Direvisi' 'Disetujui' 'Ditolak']

Apakah idsurat unik? True


In [48]:
# Mapping status
status_mapping = {
    'Direvisi': 'Sudah Revisi',
    'Disetujui': 'Disetujui',
    'Ditolak': 'Ditolak'
}

# Bentuk DataFrame final
df_sk = pd.DataFrame({
    'id_sk': df_sk_raw['idsurat'].astype('Int64'),
    'id_user': df_sk_raw['idusers'].str.strip(),
    'keterangan_sk': df_sk_raw['keterangan'].str.strip(),
    'link_dokumen_sk': df_sk_raw['link'].str.strip(),
    'status_sk': df_sk_raw['status'].map(status_mapping),
    'nomor_sk': df_sk_raw['nosurat'].str.strip(),
    'catatan_sk': df_sk_raw['catatan'].fillna('').str.strip(),
    'created_at': pd.to_datetime(df_sk_raw['created_at'])
})

# Simpan mapping untuk histori
mapping_sk = dict(zip(df_sk_raw['idsurat'], df_sk['id_sk']))

In [49]:
display(df_sk.head())

,id_sk,id_user,keterangan_sk,link_dokumen_sk,status_sk,nomor_sk,catatan_sk,created_at
0,24,U00011,Surat Permohonan Uji Kompetensi dan Penggunaan...,https://docs.google.com/document/d/1xyhc4T-FlR...,Sudah Revisi,090A/LEAP/IX/2023- 090B/LEAP/IX/2023,Lengkapi Tanggal Pelaksanaan dengan tanggal ya...,2023-09-11 17:57:49
1,26,U00011,Beasiswa Siswa GE,https://docs.google.com/document/d/1jMTK_3b57e...,Disetujui,101/LEAP/BD/X/2023,,2023-10-25 16:57:38
2,27,U00011,PERJANJIAN KERJASAMA / MEMORANDUM OF UNDERSTAN...,https://docs.google.com/document/d/1Edeb7hWaBq...,Disetujui,102/LEAP/BD/X/2023,,2023-10-26 17:37:59
3,28,U00026,Sertifikat / Sertifikat Kelas APK Private / 1 ...,https://drive.google.com/drive/folders/1JMrPJF...,Disetujui,Sertif 002a/APEX/XI/2324/02 (Page 1) dan 002b/...,,2023-11-10 16:10:04
4,29,U00011,Penawaran Pelatihan Business English ke PT. La...,https://docs.google.com/document/d/1wBK62noEJw...,Disetujui,105/LEAP/BD/XI/2023,,2023-11-13 15:41:58


In [50]:
# Cek tiap kolom
kolom_lama = ['idsurat', 'keterangan', 'link', 'idusers', 'status', 'created_at', 'nosurat', 'catatan']
kolom_baru = ['id_sk', 'id_user', 'keterangan_sk', 'link_dokumen_sk', 'status_sk', 'nomor_sk', 'catatan_sk', 'created_at']

print("=== PENGECEKAN KOLOM SURAT KELUAR ===\n")

for lama, baru in zip(kolom_lama, kolom_baru):
    print(f"--- {lama} → {baru} ---")
    print(f"  Missing: {df_sk_raw[lama].isnull().sum()}")
    if df_sk_raw[lama].dtype == 'object':
        unik = df_sk_raw[lama].dropna().unique()
        print(f"  Unique ({len(unik)}): {unik[:20]}...")  # maks 20
    else:
        print(f"  Min: {df_sk_raw[lama].min()} | Max: {df_sk_raw[lama].max()}")
    print()

=== PENGECEKAN KOLOM SURAT KELUAR ===

--- idsurat → id_sk ---
  Missing: 0
  Min: 24 | Max: 253

--- keterangan → id_user ---
  Missing: 0
  Unique (210): ['Surat Permohonan Uji Kompetensi dan Penggunaan Tempat Sebagai TUK'
 'Beasiswa Siswa GE'
 'PERJANJIAN KERJASAMA / MEMORANDUM OF UNDERSTANDING (MoU) IN-HOUSE TRAINING : Training Dasar Editing Capcut  CV. Rabbani (Fashion Retail)'
 'Sertifikat / Sertifikat Kelas APK Private / 1 Peserta / No. Sertif 002a/APEX/XI/2324/02 (Page 1)  dan 002b/APEX/XI/2324/02 (Page 2) '
 'Penawaran Pelatihan Business English ke PT. Lautan Natural Krimerindo'
 'Surat Pengantar Kajian Mitra Prakerja'
 'Konfirmasi Penerimaan Permohonan Data dan Koordinasi Pengukuran Kebutuhan Talenta Digital Indonesia berdasarkan permintaan dari KOMINFO'
 'Surat Rekomendasi CV.Rabbani' 'TOR LeapXperience Holiday Program'
 'Surat Undangan ke Sekolah LeapXperience Holiday Program'
 'Sertifikat In-house training capcut Leap x CV Rabbani '
 'NDA LMS Studiokerja (PT. EKI)' 'ToR Gr

In [51]:
pd.read_sql("DESCRIBE surat_keluar", db_new)

,Field,Type,Null,Key,Default,Extra
0,id_sk,bigint(20) unsigned,NO,PRI,None,auto_increment
1,id_user,varchar(15),YES,MUL,None,
2,keterangan_sk,text,NO,,None,
3,link_dokumen_sk,varchar(255),NO,,None,
4,status_sk,"enum('Diajukan','Sudah Revisi','Disetujui','Di...",NO,,Diajukan,
5,nomor_sk,varchar(100),NO,,None,
6,catatan_sk,text,NO,,None,
7,created_at,timestamp,NO,,current_timestamp(),


In [52]:
print("Nilai unik 'status' di data lama:")
print(df_sk_raw['status'].value_counts())

Nilai unik 'status' di data lama:
status
Disetujui    211
Direvisi       1
Ditolak        1
Name: count, dtype: int64


In [53]:
display(df_sk.head())

,id_sk,id_user,keterangan_sk,link_dokumen_sk,status_sk,nomor_sk,catatan_sk,created_at
0,24,U00011,Surat Permohonan Uji Kompetensi dan Penggunaan...,https://docs.google.com/document/d/1xyhc4T-FlR...,Sudah Revisi,090A/LEAP/IX/2023- 090B/LEAP/IX/2023,Lengkapi Tanggal Pelaksanaan dengan tanggal ya...,2023-09-11 17:57:49
1,26,U00011,Beasiswa Siswa GE,https://docs.google.com/document/d/1jMTK_3b57e...,Disetujui,101/LEAP/BD/X/2023,,2023-10-25 16:57:38
2,27,U00011,PERJANJIAN KERJASAMA / MEMORANDUM OF UNDERSTAN...,https://docs.google.com/document/d/1Edeb7hWaBq...,Disetujui,102/LEAP/BD/X/2023,,2023-10-26 17:37:59
3,28,U00026,Sertifikat / Sertifikat Kelas APK Private / 1 ...,https://drive.google.com/drive/folders/1JMrPJF...,Disetujui,Sertif 002a/APEX/XI/2324/02 (Page 1) dan 002b/...,,2023-11-10 16:10:04
4,29,U00011,Penawaran Pelatihan Business English ke PT. La...,https://docs.google.com/document/d/1wBK62noEJw...,Disetujui,105/LEAP/BD/XI/2023,,2023-11-13 15:41:58


In [54]:
# Ambil data suratkeluar_histori
df_hist_raw = pd.read_sql("SELECT * FROM suratkeluar_histori", db_old)
print("\n=== SURAT KELUAR HISTORI (lama) ===")
print("Jumlah:", len(df_hist_raw))
display(df_hist_raw.head(10))
print("\nKolom:", df_hist_raw.columns.tolist())
print("\nTipe data:")
print(df_hist_raw.dtypes)
print("\nMissing values:")
print(df_hist_raw.isnull().sum())

# Cek nilai unik status histori
print("\nNilai unik 'status':", df_hist_raw['status'].unique())


=== SURAT KELUAR HISTORI (lama) ===
Jumlah: 477


,idstatus,idsurat,status,catatan,created_at
0,7,12,Diajukan,None,2023-06-12 13:32:50
1,8,13,Diajukan,None,2023-06-12 13:33:07
2,9,14,Diajukan,None,2023-06-12 14:28:56
3,10,15,Diajukan,None,2023-06-30 15:30:35
4,11,16,Diajukan,None,2023-07-01 20:35:43
5,12,17,Diajukan,None,2023-07-03 15:00:16
6,13,18,Diajukan,None,2023-07-03 15:16:42
7,14,19,Diajukan,None,2023-07-03 15:57:42
8,15,20,Diajukan,None,2023-07-03 15:58:54
9,16,21,Diajukan,None,2023-07-03 16:00:00



Kolom: ['idstatus', 'idsurat', 'status', 'catatan', 'created_at']

Tipe data:
idstatus               int64
idsurat                int64
status                object
catatan               object
created_at    datetime64[ns]
dtype: object

Missing values:
idstatus        0
idsurat         0
status          0
catatan       246
created_at      0
dtype: int64

Nilai unik 'status': ['Diajukan' 'Revisi' 'Disetujui' 'Direvisi' '' 'Ditolak']


In [55]:
pd.read_sql("DESCRIBE verifikasi_surat_keluar", db_new)

,Field,Type,Null,Key,Default,Extra
0,id_verifikasi_surat,bigint(20) unsigned,NO,PRI,None,auto_increment
1,id_sk,bigint(20) unsigned,YES,MUL,None,
2,status_verifikasi_sk,"enum('Revisi','Diajukan','Disetujui','Ditolak')",NO,,Diajukan,
3,catatan_verifikasi_sk,text,YES,,None,
4,created_at,timestamp,NO,,current_timestamp(),


In [56]:
missing_sk = df_hist_raw[~df_hist_raw['idsurat'].isin(df_sk_raw['idsurat'])]
print(f"idsurat di histori yang tidak ada di surat_keluar: {len(missing_sk)}")
if len(missing_sk) > 0:
    print(missing_sk[['idstatus', 'idsurat']].head())

idsurat di histori yang tidak ada di surat_keluar: 35
   idstatus  idsurat
0         7       12
1         8       13
2         9       14
3        10       15
4        11       16


In [57]:
try:
    cursor_new.execute("ALTER TABLE verifikasi_surat_keluar MODIFY COLUMN catatan_verifikasi_sk text NULL;")
    db_new.commit()
    print("✅ Kolom catatan_verifikasi_sk sekarang NULLABLE.")
except Exception as e:
    print("Gagal:", e)

✅ Kolom catatan_verifikasi_sk sekarang NULLABLE.


In [58]:
try:
    cursor_new.execute("""
        ALTER TABLE verifikasi_surat_keluar 
          MODIFY COLUMN catatan_verifikasi_sk text NULL,
          MODIFY COLUMN status_verifikasi_sk enum('Revisi','Diajukan','Disetujui','Ditolak') NOT NULL DEFAULT 'Diajukan';
    """)
    db_new.commit()
    print("✅ Tabel berhasil diubah.")
except Exception as e:
    print("Gagal:", e)

✅ Tabel berhasil diubah.


In [59]:
# 1. Copy data mentah histori
df_verif = df_hist_raw.copy()

# 2. Set id_sk: ambil dari idsurat jika ada di surat_keluar, jika tidak set None (NULL di DB)
#    Gunakan tipe Int64 (Nullable Integer) agar sinkron dengan bigint(20) unsigned
id_surat_valid = set(df_sk_raw['idsurat'])
df_verif['id_sk'] = df_verif['idsurat'].apply(lambda x: x if x in id_surat_valid else None).astype('Int64')

# 3. Mapping status lama ke enum baru
#    Enum baru: 'Revisi','Diajukan','Disetujui','Ditolak'
mapping_status_verif = {
    'Diajukan': 'Diajukan',
    'Revisi': 'Revisi',
    'Disetujui': 'Disetujui',
    'Direvisi': 'Revisi',   # kita setarakan dengan Revisi
    'Ditolak': 'Ditolak',
    '': 'Diajukan'          # string kosong dianggap pengajuan baru
}
df_verif['status_verifikasi_sk'] = df_verif['status'].map(mapping_status_verif).fillna('Diajukan')

# 4. catatan_verifikasi_sk: isi None jika missing/kosong agar menjadi NULL di database
df_verif['catatan_verifikasi_sk'] = df_verif['catatan'].apply(lambda x: str(x).strip() if pd.notna(x) and str(x).strip() != '' else None)

# 5. created_at tetap sebagai datetime
df_verif['created_at'] = pd.to_datetime(df_verif['created_at'])

# 6. Pilih hanya kolom yang diperlukan (tanpa id_verifikasi_surat, biar auto_increment)
df_verif_final = df_verif[['id_sk', 'status_verifikasi_sk', 'catatan_verifikasi_sk', 'created_at']]

# 7. Cek hasil final tipe data
print("✅ verifikasi_surat_keluar final:")
display(df_verif_final.head(10))
print("\nTipe data (Aman untuk Migration):")
print(df_verif_final.dtypes)
print("\nMissing values (id_sk & catatan boleh NULL):")
print(df_verif_final.isnull().sum())

✅ verifikasi_surat_keluar final:


,id_sk,status_verifikasi_sk,catatan_verifikasi_sk,created_at
0,<NA>,Diajukan,None,2023-06-12 13:32:50
1,<NA>,Diajukan,None,2023-06-12 13:33:07
2,<NA>,Diajukan,None,2023-06-12 14:28:56
3,<NA>,Diajukan,None,2023-06-30 15:30:35
4,<NA>,Diajukan,None,2023-07-01 20:35:43
5,<NA>,Diajukan,None,2023-07-03 15:00:16
6,<NA>,Diajukan,None,2023-07-03 15:16:42
7,<NA>,Diajukan,None,2023-07-03 15:57:42
8,<NA>,Diajukan,None,2023-07-03 15:58:54
9,<NA>,Diajukan,None,2023-07-03 16:00:00



Tipe data (Aman untuk Migration):
id_sk                             Int64
status_verifikasi_sk             object
catatan_verifikasi_sk            object
created_at               datetime64[ns]
dtype: object

Missing values (id_sk & catatan boleh NULL):
id_sk                     35
status_verifikasi_sk       0
catatan_verifikasi_sk    475
created_at                 0
dtype: int64


In [60]:
pd.read_sql("DESCRIBE verifikasi_surat_keluar", db_new)

,Field,Type,Null,Key,Default,Extra
0,id_verifikasi_surat,bigint(20) unsigned,NO,PRI,None,auto_increment
1,id_sk,bigint(20) unsigned,YES,MUL,None,
2,status_verifikasi_sk,"enum('Revisi','Diajukan','Disetujui','Ditolak')",NO,,Diajukan,
3,catatan_verifikasi_sk,text,YES,,None,
4,created_at,timestamp,NO,,current_timestamp(),


In [61]:
# Ubah id_sk ke nullable integer (Int64) agar cocok dengan bigint unsigned NULL
df_verif_final['id_sk'] = df_verif_final['id_sk'].astype('Int64')

# Cek tipe data akhir
print("Tipe data setelah penyesuaian:")
print(df_verif_final.dtypes)

Tipe data setelah penyesuaian:
id_sk                             Int64
status_verifikasi_sk             object
catatan_verifikasi_sk            object
created_at               datetime64[ns]
dtype: object


In [62]:
# # Ambil skema tabel verifikasi_surat_keluar dari DB baru
# schema_ver = pd.read_sql("DESCRIBE verifikasi_surat_keluar", db_new)

# # Daftar kolom NOT NULL
# not_null_cols = schema_ver[schema_ver['Null'] == 'NO']['Field'].tolist()
# print("Kolom NOT NULL:", not_null_cols)

# # Cek apakah ada yang masih null
# for col in not_null_cols:
#     if col in df_verif_final.columns:
#         null_count = df_verif_final[col].isnull().sum()
#         if null_count > 0:
#             print(f"⚠️ {col}: {null_count} NULL — akan diisi default")
#             # Ambil tipe data dari skema
#             col_type = schema_ver[schema_ver['Field'] == col]['Type'].values[0]
#             # Tentukan default berdasarkan tipe
#             if 'int' in col_type:
#                 df_verif_final[col] = df_verif_final[col].fillna(0).astype(int)
#             elif 'varchar' in col_type or 'text' in col_type:
#                 df_verif_final[col] = df_verif_final[col].fillna('')
#             elif 'enum' in col_type:
#                 # Ambil nilai enum pertama sebagai default
#                 # Contoh: enum('Revisi','Diterima','Disetujui','Ditolak')
#                 # Kita bisa ambil 'Diterima' atau dari mapping. Asumsi aman: ambil enum pertama.
#                 enum_vals = col_type.split("'")[1::2]  # ambil string dalam kutip
#                 default_enum = enum_vals[0] if enum_vals else ''
#                 df_verif_final[col] = df_verif_final[col].fillna(default_enum)
#             elif 'date' in col_type or 'time' in col_type:
#                 df_verif_final[col] = pd.to_datetime(df_verif_final[col]).fillna(pd.Timestamp('1970-01-01'))
#             else:
#                 df_verif_final[col] = df_verif_final[col].fillna('')
#         else:
#             print(f"✅ {col}: aman")
#     else:
#         print(f"⚠️ Kolom {col} tidak ditemukan di dataframe, perlu ditambahkan!")

## 3. Surat Tugas dan Surat Tugas Anggota

In [63]:
# Data surattugas dari DB lama
df_st_raw = pd.read_sql("SELECT * FROM surattugas", db_old)
print("=== SURAT TUGAS (lama) ===")
print("Jumlah:", len(df_st_raw))
display(df_st_raw.head(5))
print("\nKolom:", df_st_raw.columns.tolist())
print("\nTipe data:")
print(df_st_raw.dtypes)
print("\nMissing values:")
print(df_st_raw.isnull().sum())
print("\nNilai unik 'status':", df_st_raw['status'].unique())
print("Nilai unik 'jenis':", df_st_raw['jenis'].unique())

=== SURAT TUGAS (lama) ===
Jumlah: 135


,idsurat,acara,undangan,waktu,lokasi,jenis,status,idusers,created_at,nosurat,catatan,link,linklaporan,notelaporan,ket,notebatal
0,20,Pertemuan rutin di bulan Juli 2023 forum Laras...,Larasdikdudi,"<p><span class=""fontstyle0"">Hari, Tanggal : Ka...",SMK Teknik PAL Surabaya Jalan Ujung Surabaya,Offline,Disetujui,U00015,2023-07-25 09:56:19,023/LEAP/ST/VII/2023,,https://docs.google.com/document/d/1rILu6FE8Bd...,https://docs.google.com/document/d/1IAhWT4XjbX...,done/laporan sudah diisi,None,None
1,21,Temu Warga RT 01 RW 08,Pengurus RT 01/ RW08,"<p>Rabu, 23 Agustus 2023<br />jam 19.00-selesa...",Balai RW,Offline,Disetujui,U00015,2023-08-22 17:43:48,024/LEAP/ST/VIII/2023,,https://docs.google.com/document/d/1OQxpdKYfDF...,https://docs.google.com/document/d/1XrvRmYckBw...,None,None,None
2,22,TEDxSurabaya Translators:\r\n\r\n1. Simulasi C...,TEDxSurabaya,<p>1. Simulasi Coaching:&nbsp;15 September 202...,Topic: Connect to TEDxSurabaya Join Zoom Meeti...,Online,Disetujui,U00016,2023-09-13 13:01:52,025/LEAP/ST/VIII/2023,,https://docs.google.com/document/d/10Zi6g0R9RR...,https://docs.google.com/document/d/1pa9G3E3eUa...,laporan kegiatan done,None,None
3,23,Pertemuan Rutin Larasdikdudi bulan September 2023,https://drive.google.com/drive/u/3/folders/1OI...,"<p><span class=""fontstyle0"">Hari, Tanggal : Ra...",AULA SMK Negeri 2 Surabaya Jalan Tentara Genie...,Offline,Disetujui,U00015,2023-09-19 13:54:26,028/LEAP/ST/IX/2023,,https://docs.google.com/document/d/1wMOD-N6oMQ...,https://docs.google.com/document/d/1xMv7AK2L9S...,done,None,None
4,24,Sosialisasi Kerjasama LKP dan PKBM dengan Seme...,Dinas Pendidikan Kota Surabaya (bu Hilda),"<table class=""NormalTable"">\r\n<tbody>\r\n<tr>...","Tempat : Ruang Bung Tomo, Dinas Pendidikan Kot...",Offline,Disetujui,U00015,2023-09-19 13:55:50,027/LEAP/ST/IX/2023,,https://docs.google.com/document/d/1LBx4WpqXcY...,https://docs.google.com/document/d/1jJdGMkonp_...,None,None,None



Kolom: ['idsurat', 'acara', 'undangan', 'waktu', 'lokasi', 'jenis', 'status', 'idusers', 'created_at', 'nosurat', 'catatan', 'link', 'linklaporan', 'notelaporan', 'ket', 'notebatal']

Tipe data:
idsurat                 int64
acara                  object
undangan               object
waktu                  object
lokasi                 object
jenis                  object
status                 object
idusers                object
created_at     datetime64[ns]
nosurat                object
catatan                object
link                   object
linklaporan            object
notelaporan            object
ket                    object
notebatal              object
dtype: object

Missing values:
idsurat          0
acara            0
undangan         0
waktu            0
lokasi           0
jenis            0
status           0
idusers          0
created_at       0
nosurat          0
catatan          0
link             0
linklaporan      0
notelaporan     58
ket            132
notebata

In [64]:
# Struktur tabel surat_tugas di DB baru
print("\nStruktur surat_tugas (db_old):")
pd.read_sql("DESCRIBE surattugas", db_old)


Struktur surat_tugas (db_old):


,Field,Type,Null,Key,Default,Extra
0,idsurat,int(11),NO,PRI,None,auto_increment
1,acara,text,NO,,None,
2,undangan,text,NO,,'',
3,waktu,text,NO,,'',
4,lokasi,text,NO,,'',
5,jenis,varchar(100),NO,,,
6,status,varchar(100),NO,,,
7,idusers,varchar(6),NO,MUL,,
8,created_at,datetime,NO,,current_timestamp(),
9,nosurat,varchar(50),YES,,None,


In [65]:
print("\nStruktur surat_tugas (db_new):")
pd.read_sql("DESCRIBE surat_tugas", db_new)


Struktur surat_tugas (db_new):


,Field,Type,Null,Key,Default,Extra
0,id_st,bigint(20) unsigned,NO,PRI,None,auto_increment
1,id_user,varchar(15),YES,MUL,None,
2,acara,varchar(255),NO,,None,
3,undangan,varchar(255),NO,,None,
4,waktu_acara,timestamp,NO,,None,
5,lokasi_acara,varchar(255),NO,,None,
6,jenis_kegiatan,"enum('Offline','Online')",NO,,None,
7,status_st,"enum('Diajukan','Disetujui','Dibatalkan')",NO,,Diajukan,
8,nomor_st,varchar(100),NO,,None,
9,catatan_st,text,NO,,None,


In [66]:
# Lihat beberapa baris dan nilai unik
print(df_st_raw[['jenis', 'status']].head(5))
print("\nNilai unik jenis:", df_st_raw['jenis'].unique())
print("Nilai unik status:", df_st_raw['status'].unique())

     jenis     status
0  Offline  Disetujui
1  Offline  Disetujui
2   Online  Disetujui
3  Offline  Disetujui
4  Offline  Disetujui

Nilai unik jenis: ['Offline' 'Online']
Nilai unik status: ['Disetujui' 'Dibatalkan']


In [67]:
# 1. Konversi waktu (teks) ke datetime, error='coerce' agar yang gagal jadi NaT
df_st_raw['waktu_dt'] = pd.to_datetime(df_st_raw['waktu'], errors='coerce')

# 2. Bentuk DataFrame final
df_st = pd.DataFrame({
    'id_st': df_st_raw['idsurat'],                # kita pertahankan ID integer dari lama
    'id_user': df_st_raw['idusers'].str.strip(),
    'acara': df_st_raw['acara'].str.strip(),
    'undangan': df_st_raw['undangan'].str.strip(),
    'waktu_acara': df_st_raw['waktu_dt'],         # sudah datetime
    'lokasi_acara': df_st_raw['lokasi'].str.strip(),
    'jenis_kegiatan': df_st_raw['jenis'].str.strip(),  # sudah 'Offline'/'Online'
    'status_st': df_st_raw['status'].str.strip(),      # sudah 'Disetujui'/'Dibatalkan'
    'nomor_st': df_st_raw['nosurat'].fillna('').str.strip(),
    'catatan_st': df_st_raw['catatan'].fillna('').str.strip(),
    'link_st': df_st_raw['link'].fillna('').str.strip(),
    'link_laporan': df_st_raw['linklaporan'].fillna('').str.strip(),
    'catatan_laporan': df_st_raw['notelaporan'].fillna('').str.strip(),
    'keterangan_st': df_st_raw['ket'].fillna('').str.strip(),
    'catatan_pembatalan': df_st_raw['notebatal'].fillna('').str.strip(),
    'created_at': pd.to_datetime(df_st_raw['created_at'])
})

# 3. Cek hasil
print("✅ df_st siap:")
display(df_st.head())
print("Tipe data:")
print(df_st.dtypes)
print("Missing values:")
print(df_st.isnull().sum())

# 4. Simpan mapping id_st (lama -> baru, sebenarnya sama)
mapping_st = dict(zip(df_st_raw['idsurat'], df_st['id_st']))
print("Mapping surat_tugas (lama -> baru):", list(mapping_st.items())[:5])


✅ df_st siap:


,id_st,id_user,acara,undangan,waktu_acara,lokasi_acara,jenis_kegiatan,status_st,nomor_st,catatan_st,link_st,link_laporan,catatan_laporan,keterangan_st,catatan_pembatalan,created_at
0,20,U00015,Pertemuan rutin di bulan Juli 2023 forum Laras...,Larasdikdudi,NaT,SMK Teknik PAL Surabaya Jalan Ujung Surabaya,Offline,Disetujui,023/LEAP/ST/VII/2023,,https://docs.google.com/document/d/1rILu6FE8Bd...,https://docs.google.com/document/d/1IAhWT4XjbX...,done/laporan sudah diisi,,,2023-07-25 09:56:19
1,21,U00015,Temu Warga RT 01 RW 08,Pengurus RT 01/ RW08,NaT,Balai RW,Offline,Disetujui,024/LEAP/ST/VIII/2023,,https://docs.google.com/document/d/1OQxpdKYfDF...,https://docs.google.com/document/d/1XrvRmYckBw...,,,,2023-08-22 17:43:48
2,22,U00016,TEDxSurabaya Translators:\r\n\r\n1. Simulasi C...,TEDxSurabaya,NaT,Topic: Connect to TEDxSurabaya Join Zoom Meeti...,Online,Disetujui,025/LEAP/ST/VIII/2023,,https://docs.google.com/document/d/10Zi6g0R9RR...,https://docs.google.com/document/d/1pa9G3E3eUa...,laporan kegiatan done,,,2023-09-13 13:01:52
3,23,U00015,Pertemuan Rutin Larasdikdudi bulan September 2023,https://drive.google.com/drive/u/3/folders/1OI...,NaT,AULA SMK Negeri 2 Surabaya Jalan Tentara Genie...,Offline,Disetujui,028/LEAP/ST/IX/2023,,https://docs.google.com/document/d/1wMOD-N6oMQ...,https://docs.google.com/document/d/1xMv7AK2L9S...,done,,,2023-09-19 13:54:26
4,24,U00015,Sosialisasi Kerjasama LKP dan PKBM dengan Seme...,Dinas Pendidikan Kota Surabaya (bu Hilda),NaT,"Tempat : Ruang Bung Tomo, Dinas Pendidikan Kot...",Offline,Disetujui,027/LEAP/ST/IX/2023,,https://docs.google.com/document/d/1LBx4WpqXcY...,https://docs.google.com/document/d/1jJdGMkonp_...,,,,2023-09-19 13:55:50


Tipe data:
id_st                          int64
id_user                       object
acara                         object
undangan                      object
waktu_acara           datetime64[ns]
lokasi_acara                  object
jenis_kegiatan                object
status_st                     object
nomor_st                      object
catatan_st                    object
link_st                       object
link_laporan                  object
catatan_laporan               object
keterangan_st                 object
catatan_pembatalan            object
created_at            datetime64[ns]
dtype: object
Missing values:
id_st                   0
id_user                 0
acara                   0
undangan                0
waktu_acara           135
lokasi_acara            0
jenis_kegiatan          0
status_st               0
nomor_st                0
catatan_st              0
link_st                 0
link_laporan            0
catatan_laporan         0
keterangan_st           0
cat

In [68]:
print("Contoh isi kolom waktu:")
print(df_st_raw['waktu'].head(10))
print("\nJumlah missing:", df_st_raw['waktu'].isnull().sum())

Contoh isi kolom waktu:
0    <p><span class="fontstyle0">Hari, Tanggal : Ka...
1    <p>Rabu, 23 Agustus 2023<br />jam 19.00-selesa...
2    <p>1. Simulasi Coaching:&nbsp;15 September 202...
3    <p><span class="fontstyle0">Hari, Tanggal : Ra...
4    <table class="NormalTable">\r\n<tbody>\r\n<tr>...
5    <p>Jum'at 8 September 2023</p>\r\n<p>Pukul 14....
6    <p>Jum'at, 22 September 2023 jam 15.00-16.30 W...
7    <p>hari, tanggal : Selasa, 26 September 2023</...
8    <p>hari, tanggal : Selasa, 26 September 2023</...
9    <p>Minggu, 8 Oktober 2023</p>\r\n<p>Jam 17.15-...
Name: waktu, dtype: object

Jumlah missing: 0


In [69]:
import re
import pandas as pd
from datetime import datetime

# Fungsi untuk mengekstrak tanggal dari teks
def extract_date(text):
    if pd.isna(text) or not isinstance(text, str):
        return None
    
    # Pola tanggal: "27 juli 2023", "23 Agustus 2023", "15 September 2023", "8 Oktober 2023", dll.
    # Nama bulan Indonesia
    bulan_id = r'(Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)'
    # Pola: (hari) (bulan) (tahun) - hari bisa 1-2 digit
    pattern = rf'(\d{{1,2}})\s+{bulan_id}\s+(\d{{4}})'
    
    match = re.search(pattern, text, re.IGNORECASE)
    if match:
        hari = int(match.group(1))
        bulan_str = match.group(2).lower()
        tahun = int(match.group(3))
        # Mapping bulan lowercase ke angka
        bulan_map = {
            'januari':1,'februari':2,'maret':3,'april':4,'mei':5,'juni':6,
            'juli':7,'agustus':8,'september':9,'oktober':10,'november':11,'desember':12
        }
        bulan = bulan_map.get(bulan_str)
        if bulan:
            try:
                return pd.Timestamp(year=tahun, month=bulan, day=hari)
            except:
                pass
    return None

# Terapkan fungsi
df_st_raw['waktu_dt'] = df_st_raw['waktu'].apply(extract_date)

# Hitung ulang yang masih kosong
gagal = df_st_raw['waktu_dt'].isnull()
print(f"Masih gagal: {gagal.sum()} dari {len(df_st_raw)}")

# Untuk yang gagal, fallback ke created_at
df_st_raw.loc[gagal, 'waktu_dt'] = pd.to_datetime(df_st_raw.loc[gagal, 'created_at'])

print("Setelah fallback:")
print(df_st_raw['waktu_dt'].isnull().sum(), "missing (harusnya 0)")
print("\nContoh hasil:")
print(df_st_raw[['waktu_dt']].head())

Masih gagal: 14 dari 135
Setelah fallback:
0 missing (harusnya 0)

Contoh hasil:
    waktu_dt
0 2023-07-27
1 2023-08-23
2 2023-09-15
3 2023-09-27
4 2023-09-20


In [70]:
import re

# Mapping bulan Indonesia (jika belum ada)
bulan_map = {
    'januari': 1, 'februari': 2, 'maret': 3,
    'april': 4, 'mei': 5, 'juni': 6,
    'juli': 7, 'agustus': 8, 'september': 9,
    'oktober': 10, 'november': 11, 'desember': 12
}

def extract_date_id(text):
    if pd.isna(text) or not isinstance(text, str):
        return None
    pattern = r'(\d{1,2})\s+(Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember),?\s+(\d{4})'
    match = re.search(pattern, text, re.IGNORECASE)
    if match:
        hari = int(match.group(1))
        bulan = bulan_map[match.group(2).lower()]
        tahun = int(match.group(3))
        try:
            return pd.Timestamp(year=tahun, month=bulan, day=hari)
        except:
            return None
    return None

# Ekstrak tanggal dari kolom 'waktu'
df_st_raw['waktu_dt'] = df_st_raw['waktu'].apply(extract_date_id)

# Fallback: gunakan created_at untuk yang masih kosong
mask_null = df_st_raw['waktu_dt'].isnull()
print(f"Fallback untuk {mask_null.sum()} baris (pakai created_at).")
df_st_raw.loc[mask_null, 'waktu_dt'] = pd.to_datetime(df_st_raw.loc[mask_null, 'created_at'])

# Sekarang bangun DataFrame final surat_tugas
df_st = pd.DataFrame({
    'id_st': df_st_raw['idsurat'],
    'id_user': df_st_raw['idusers'].str.strip(),
    'acara': df_st_raw['acara'].str.strip(),
    'undangan': df_st_raw['undangan'].str.strip(),
    'waktu_acara': df_st_raw['waktu_dt'],
    'lokasi_acara': df_st_raw['lokasi'].str.strip(),
    'jenis_kegiatan': df_st_raw['jenis'].str.strip(),
    'status_st': df_st_raw['status'].str.strip(),
    'nomor_st': df_st_raw['nosurat'].fillna('').str.strip(),
    'catatan_st': df_st_raw['catatan'].fillna('').str.strip(),
    'link_st': df_st_raw['link'].fillna('').str.strip(),
    'link_laporan': df_st_raw['linklaporan'].fillna('').str.strip(),
    'catatan_laporan': df_st_raw['notelaporan'].fillna('').str.strip(),
    'keterangan_st': df_st_raw['ket'].fillna('').str.strip(),
    'catatan_pembatalan': df_st_raw['notebatal'].fillna('').str.strip(),
    'created_at': pd.to_datetime(df_st_raw['created_at'])
})

print("✅ df_st siap:")
display(df_st.head())
print("Tipe data:")
print(df_st.dtypes)
print("Missing values:")
print(df_st.isnull().sum())

# Simpan mapping id_st (lama -> baru)
mapping_st = dict(zip(df_st_raw['idsurat'], df_st['id_st']))
print("Mapping surat_tugas (lama -> baru):", list(mapping_st.items())[:5])

Fallback untuk 14 baris (pakai created_at).
✅ df_st siap:


,id_st,id_user,acara,undangan,waktu_acara,lokasi_acara,jenis_kegiatan,status_st,nomor_st,catatan_st,link_st,link_laporan,catatan_laporan,keterangan_st,catatan_pembatalan,created_at
0,20,U00015,Pertemuan rutin di bulan Juli 2023 forum Laras...,Larasdikdudi,2023-07-27,SMK Teknik PAL Surabaya Jalan Ujung Surabaya,Offline,Disetujui,023/LEAP/ST/VII/2023,,https://docs.google.com/document/d/1rILu6FE8Bd...,https://docs.google.com/document/d/1IAhWT4XjbX...,done/laporan sudah diisi,,,2023-07-25 09:56:19
1,21,U00015,Temu Warga RT 01 RW 08,Pengurus RT 01/ RW08,2023-08-23,Balai RW,Offline,Disetujui,024/LEAP/ST/VIII/2023,,https://docs.google.com/document/d/1OQxpdKYfDF...,https://docs.google.com/document/d/1XrvRmYckBw...,,,,2023-08-22 17:43:48
2,22,U00016,TEDxSurabaya Translators:\r\n\r\n1. Simulasi C...,TEDxSurabaya,2023-09-15,Topic: Connect to TEDxSurabaya Join Zoom Meeti...,Online,Disetujui,025/LEAP/ST/VIII/2023,,https://docs.google.com/document/d/10Zi6g0R9RR...,https://docs.google.com/document/d/1pa9G3E3eUa...,laporan kegiatan done,,,2023-09-13 13:01:52
3,23,U00015,Pertemuan Rutin Larasdikdudi bulan September 2023,https://drive.google.com/drive/u/3/folders/1OI...,2023-09-27,AULA SMK Negeri 2 Surabaya Jalan Tentara Genie...,Offline,Disetujui,028/LEAP/ST/IX/2023,,https://docs.google.com/document/d/1wMOD-N6oMQ...,https://docs.google.com/document/d/1xMv7AK2L9S...,done,,,2023-09-19 13:54:26
4,24,U00015,Sosialisasi Kerjasama LKP dan PKBM dengan Seme...,Dinas Pendidikan Kota Surabaya (bu Hilda),2023-09-20,"Tempat : Ruang Bung Tomo, Dinas Pendidikan Kot...",Offline,Disetujui,027/LEAP/ST/IX/2023,,https://docs.google.com/document/d/1LBx4WpqXcY...,https://docs.google.com/document/d/1jJdGMkonp_...,,,,2023-09-19 13:55:50


Tipe data:
id_st                          int64
id_user                       object
acara                         object
undangan                      object
waktu_acara           datetime64[ns]
lokasi_acara                  object
jenis_kegiatan                object
status_st                     object
nomor_st                      object
catatan_st                    object
link_st                       object
link_laporan                  object
catatan_laporan               object
keterangan_st                 object
catatan_pembatalan            object
created_at            datetime64[ns]
dtype: object
Missing values:
id_st                 0
id_user               0
acara                 0
undangan              0
waktu_acara           0
lokasi_acara          0
jenis_kegiatan        0
status_st             0
nomor_st              0
catatan_st            0
link_st               0
link_laporan          0
catatan_laporan       0
keterangan_st         0
catatan_pembatalan    0
created

In [71]:
display(df_st.head())

,id_st,id_user,acara,undangan,waktu_acara,lokasi_acara,jenis_kegiatan,status_st,nomor_st,catatan_st,link_st,link_laporan,catatan_laporan,keterangan_st,catatan_pembatalan,created_at
0,20,U00015,Pertemuan rutin di bulan Juli 2023 forum Laras...,Larasdikdudi,2023-07-27,SMK Teknik PAL Surabaya Jalan Ujung Surabaya,Offline,Disetujui,023/LEAP/ST/VII/2023,,https://docs.google.com/document/d/1rILu6FE8Bd...,https://docs.google.com/document/d/1IAhWT4XjbX...,done/laporan sudah diisi,,,2023-07-25 09:56:19
1,21,U00015,Temu Warga RT 01 RW 08,Pengurus RT 01/ RW08,2023-08-23,Balai RW,Offline,Disetujui,024/LEAP/ST/VIII/2023,,https://docs.google.com/document/d/1OQxpdKYfDF...,https://docs.google.com/document/d/1XrvRmYckBw...,,,,2023-08-22 17:43:48
2,22,U00016,TEDxSurabaya Translators:\r\n\r\n1. Simulasi C...,TEDxSurabaya,2023-09-15,Topic: Connect to TEDxSurabaya Join Zoom Meeti...,Online,Disetujui,025/LEAP/ST/VIII/2023,,https://docs.google.com/document/d/10Zi6g0R9RR...,https://docs.google.com/document/d/1pa9G3E3eUa...,laporan kegiatan done,,,2023-09-13 13:01:52
3,23,U00015,Pertemuan Rutin Larasdikdudi bulan September 2023,https://drive.google.com/drive/u/3/folders/1OI...,2023-09-27,AULA SMK Negeri 2 Surabaya Jalan Tentara Genie...,Offline,Disetujui,028/LEAP/ST/IX/2023,,https://docs.google.com/document/d/1wMOD-N6oMQ...,https://docs.google.com/document/d/1xMv7AK2L9S...,done,,,2023-09-19 13:54:26
4,24,U00015,Sosialisasi Kerjasama LKP dan PKBM dengan Seme...,Dinas Pendidikan Kota Surabaya (bu Hilda),2023-09-20,"Tempat : Ruang Bung Tomo, Dinas Pendidikan Kot...",Offline,Disetujui,027/LEAP/ST/IX/2023,,https://docs.google.com/document/d/1LBx4WpqXcY...,https://docs.google.com/document/d/1jJdGMkonp_...,,,,2023-09-19 13:55:50


In [72]:
df_stu_raw = pd.read_sql("SELECT * FROM surattugas_users", db_old)
print("=== SURAT TUGAS USERS (lama) ===")
print("Jumlah:", len(df_stu_raw))
display(df_stu_raw.head(10))
print("\nKolom:", df_stu_raw.columns.tolist())
print("\nTipe data:")
print(df_stu_raw.dtypes)
print("\nMissing values:")
print(df_stu_raw.isnull().sum())

=== SURAT TUGAS USERS (lama) ===
Jumlah: 304


,idsu,idsurat,idusers
0,15,6,U00012
1,16,6,U00003
2,17,7,U00026
3,18,7,U00012
4,19,8,U00026
5,20,8,U00018
6,21,9,U00026
7,22,9,U00016
8,23,9,U00018
9,24,10,U00011



Kolom: ['idsu', 'idsurat', 'idusers']

Tipe data:
idsu        int64
idsurat     int64
idusers    object
dtype: object

Missing values:
idsu       0
idsurat    0
idusers    0
dtype: int64


In [73]:
# Bentuk DataFrame surat_tugas_anggota
df_sta = pd.DataFrame({
    'id_st': df_stu_raw['idsurat'],       # sama dengan id_st di surat_tugas
    'id_user': df_stu_raw['idusers'].str.strip()
})

print("✅ surat_tugas_anggota siap:")
display(df_sta.head())
print("Tipe data:")
print(df_sta.dtypes)
print("Missing values:")
print(df_sta.isnull().sum())


✅ surat_tugas_anggota siap:


,id_st,id_user
0,6,U00012
1,6,U00003
2,7,U00026
3,7,U00012
4,8,U00026


Tipe data:
id_st       int64
id_user    object
dtype: object
Missing values:
id_st      0
id_user    0
dtype: int64


In [74]:
invalid_id = df_sta[
    ~df_sta['id_st'].isin(df_sta['id_st'])
]

print(invalid_id)

Empty DataFrame
Columns: [id_st, id_user]
Index: []


In [75]:
print(invalid_id['id_st'].unique())

[]


In [76]:
df_sta = df_sta[
    df_sta['id_st'].isin(df_sta['id_st'])
]

## SOP dan SOP Kategori

In [77]:
# Ambil data sopkategori dari DB lama
df_kat_raw = pd.read_sql("SELECT * FROM sopkategori", db_old)
print("Data sopkategori lama:")
display(df_kat_raw.head())

# Buat ID integer urut (1,2,3,...) untuk id_sop_kategori baru
old_ids = df_kat_raw['idsopkategori'].tolist()
mapping_kat = {old: i+1 for i, old in enumerate(old_ids)}

print("Mapping kategori (lama -> baru):")
for k, v in mapping_kat.items():
    print(f"  {k} -> {v}")

# Buat DataFrame sop_kategori
df_sop_kategori = pd.DataFrame({
    'id_sop_kategori': range(1, len(df_kat_raw) + 1),
    'nama_kategori_sop': df_kat_raw['nama'].fillna('').str.strip()
})

# Simpan ke dictionary fase3
fase3_data = {}
fase3_data['sop_kategori'] = df_sop_kategori
print("✅ sop_kategori selesai")

Data sopkategori lama:


,idsopkategori,nama
0,K00001,Kelas
1,K00002,HR / GA
2,K00003,test


Mapping kategori (lama -> baru):
  K00001 -> 1
  K00002 -> 2
  K00003 -> 3
✅ sop_kategori selesai


In [78]:
# =================================================
# PROSES TABEL: sop (tanpa id_sop)
# =================================================

# 1. Ambil data mentah dari DB lama
df_sop_raw = pd.read_sql("SELECT * FROM sop", db_old)
print("Data SOP mentah:")
display(df_sop_raw.head())

# 2. Mapping kolom, bersihkan, dan ubah tipe data
df_sop = pd.DataFrame({
    'judul_sop':           df_sop_raw['judulsop'].fillna('').str.strip(),
    'link_dokumen_sop':    df_sop_raw['link'].fillna('').str.strip(),
    'created_at':          pd.to_datetime(df_sop_raw['created_at']),
    'id_sop_kategori':     df_sop_raw['idsopkategori'].map(mapping_kat)   # FK ke sop_kategori
})

# 3. Cek null & duplikat (meski tidak ada PK, kita lihat duplikat baris)
print("\nMissing values:")
print(df_sop.isnull().sum())
print(f"\nJumlah baris duplikat: {df_sop.duplicated().sum()}")

# 4. Simpan ke dictionary fase3
fase3_data['sop'] = df_sop

print("\n✅ DataFrame sop final (tanpa id_sop):")
display(df_sop.head())
print("\nTipe data:")
print(df_sop.dtypes)

Data SOP mentah:


,idsop,judulsop,link,created_at,idsopkategori
0,s00001,Akhir Penggunaan Kelas English (19.15 WIB),https://drive.google.com/file/d/1V0MpB7ctzPHe6...,2023-07-07 08:52:48,K00001
1,s00002,Penerimaan Surat Masuk,https://drive.google.com/file/d/1G8lO33yU7MufC...,2023-11-22 15:27:06,K00002
2,s00003,Pengajuan Surat Keluar,https://drive.google.com/file/d/14I7wYSk62WJAv...,2023-11-22 15:27:55,K00002
3,s00004,Pengajuan Surat Tugas,https://drive.google.com/file/d/1R-wKFqQeVSz62...,2023-11-22 15:28:27,K00002



Missing values:
judul_sop           0
link_dokumen_sop    0
created_at          0
id_sop_kategori     0
dtype: int64

Jumlah baris duplikat: 0

✅ DataFrame sop final (tanpa id_sop):


,judul_sop,link_dokumen_sop,created_at,id_sop_kategori
0,Akhir Penggunaan Kelas English (19.15 WIB),https://drive.google.com/file/d/1V0MpB7ctzPHe6...,2023-07-07 08:52:48,1
1,Penerimaan Surat Masuk,https://drive.google.com/file/d/1G8lO33yU7MufC...,2023-11-22 15:27:06,2
2,Pengajuan Surat Keluar,https://drive.google.com/file/d/14I7wYSk62WJAv...,2023-11-22 15:27:55,2
3,Pengajuan Surat Tugas,https://drive.google.com/file/d/1R-wKFqQeVSz62...,2023-11-22 15:28:27,2



Tipe data:
judul_sop                   object
link_dokumen_sop            object
created_at          datetime64[ns]
id_sop_kategori              int64
dtype: object


In [79]:
# # Cek panjang maksimum kolom yang bermasalah
# df_sk = pd.read_sql("SELECT nomor_sk FROM surat_keluar", db_new)
# max_nomor = df_sk['nomor_sk'].dropna().str.len().max()
# print(f"Panjang maksimum nomor_sk: {max_nomor}")

# df_st = pd.read_sql("SELECT lokasi_acara FROM surat_tugas", db_new)
# max_lokasi = df_st['lokasi_acara'].dropna().str.len().max()
# print(f"Panjang maksimum lokasi_acara: {max_lokasi}")

In [80]:
# # Cek panjang maksimum kolom yang bermasalah
# df_sk = pd.read_sql("SELECT nosurat FROM suratkeluar", db_old)
# max_nomor = df_sk['nosurat'].dropna().str.len().max()
# print(f"Panjang maksimum nosurat: {max_nomor}")

# df_st = pd.read_sql("SELECT lokasi FROM surattugas", db_old)
# max_lokasi = df_st['lokasi'].dropna().str.len().max()
# print(f"Panjang maksimum lokasi: {max_lokasi}")

## Fixing

In [81]:
display(df_st.head())

,id_st,id_user,acara,undangan,waktu_acara,lokasi_acara,jenis_kegiatan,status_st,nomor_st,catatan_st,link_st,link_laporan,catatan_laporan,keterangan_st,catatan_pembatalan,created_at
0,20,U00015,Pertemuan rutin di bulan Juli 2023 forum Laras...,Larasdikdudi,2023-07-27,SMK Teknik PAL Surabaya Jalan Ujung Surabaya,Offline,Disetujui,023/LEAP/ST/VII/2023,,https://docs.google.com/document/d/1rILu6FE8Bd...,https://docs.google.com/document/d/1IAhWT4XjbX...,done/laporan sudah diisi,,,2023-07-25 09:56:19
1,21,U00015,Temu Warga RT 01 RW 08,Pengurus RT 01/ RW08,2023-08-23,Balai RW,Offline,Disetujui,024/LEAP/ST/VIII/2023,,https://docs.google.com/document/d/1OQxpdKYfDF...,https://docs.google.com/document/d/1XrvRmYckBw...,,,,2023-08-22 17:43:48
2,22,U00016,TEDxSurabaya Translators:\r\n\r\n1. Simulasi C...,TEDxSurabaya,2023-09-15,Topic: Connect to TEDxSurabaya Join Zoom Meeti...,Online,Disetujui,025/LEAP/ST/VIII/2023,,https://docs.google.com/document/d/10Zi6g0R9RR...,https://docs.google.com/document/d/1pa9G3E3eUa...,laporan kegiatan done,,,2023-09-13 13:01:52
3,23,U00015,Pertemuan Rutin Larasdikdudi bulan September 2023,https://drive.google.com/drive/u/3/folders/1OI...,2023-09-27,AULA SMK Negeri 2 Surabaya Jalan Tentara Genie...,Offline,Disetujui,028/LEAP/ST/IX/2023,,https://docs.google.com/document/d/1wMOD-N6oMQ...,https://docs.google.com/document/d/1xMv7AK2L9S...,done,,,2023-09-19 13:54:26
4,24,U00015,Sosialisasi Kerjasama LKP dan PKBM dengan Seme...,Dinas Pendidikan Kota Surabaya (bu Hilda),2023-09-20,"Tempat : Ruang Bung Tomo, Dinas Pendidikan Kot...",Offline,Disetujui,027/LEAP/ST/IX/2023,,https://docs.google.com/document/d/1LBx4WpqXcY...,https://docs.google.com/document/d/1jJdGMkonp_...,,,,2023-09-19 13:55:50


In [82]:
# # Surat Keluar
# df_sk_raw = pd.read_sql("SELECT * FROM suratkeluar", db_old)
# # Generate ID integer 1,2,3...
# old_sk_ids = df_sk_raw['idsurat'].tolist()
# mapping_sk = {old: i+1 for i, old in enumerate(old_sk_ids)}
# print("Mapping surat_keluar:", len(mapping_sk))

# # Verifikasi (histori) surat keluar
# df_ver_raw = pd.read_sql("SELECT * FROM suratkeluar_histori", db_old)

In [83]:
# df_sk = pd.DataFrame({
#     'id_sk': [mapping_sk[old] for old in old_sk_ids],
#     'id_user': df_sk_raw['idusers'].fillna('').str.strip(),
#     'keterangan_sk': df_sk_raw['keterangan'].fillna(''),
#     'link_dokumen_sk': df_sk_raw['link'].fillna(''),
#     'status_sk': df_sk_raw['status'],  # nanti divalidasi enum
#     'nomor_sk': df_sk_raw['nosurat'].fillna('').str.strip(),
#     'catatan_sk': df_sk_raw['catatan'].fillna(''),
#     'created_at': pd.to_datetime(df_sk_raw['created_at'])
# })

# # Validasi enum status_sk
# valid_status_sk = ['Diajukan','Sudah Revisi','Disetujui','Ditolak']
# df_sk['status_sk'] = df_sk['status_sk'].apply(lambda x: x if x in valid_status_sk else 'Diajukan')

# # Pastikan tidak ada NULL di kolom NOT NULL (cek schema dulu)
# # Untuk saat ini kita asumsikan yang NOT NULL sudah terisi, jika ada kita isi default nanti.

In [84]:
# # Mapping id status lama -> integer baru (untuk PK verifikasi)
# old_ver_ids = df_ver_raw['idstatus'].tolist()
# mapping_ver = {old: i+1 for i, old in enumerate(old_ver_ids)}

# df_ver = pd.DataFrame({
#     'id_verifikasi_surat': range(1, len(df_ver_raw)+1),
#     'id_sk': df_ver_raw['idsurat'].map(mapping_sk),  # FK ke surat_keluar
#     'status_verifikasi_sk': df_ver_raw['status'],
#     'catatan_verifikasi_sk': df_ver_raw['catatan'].fillna(''),
#     'created_at': pd.to_datetime(df_ver_raw['created_at'])
# })
    
# # Validasi enum
# valid_ver_status = ['Revisi','Diterima','Disetujui','Ditolak']
# df_ver['status_verifikasi_sk'] = df_ver['status_verifikasi_sk'].apply(lambda x: x if x in valid_ver_status else 'Diterima')

# # **Pastikan semua NOT NULL terisi**
# # Gunakan kode anti-NaN yang kemarin
# schema_ver = pd.read_sql("DESCRIBE verifikasi_surat_keluar", db_new)
# for _, row in schema_ver.iterrows():
#     col = row['Field']
#     if row['Null'] == 'NO' and col in df_ver.columns:
#         if df_ver[col].isnull().any():
#             col_type = row['Type']
#             if 'int' in col_type:
#                 df_ver[col] = df_ver[col].fillna(0)
#             elif 'char' in col_type or 'text' in col_type:
#                 df_ver[col] = df_ver[col].fillna('')
#             elif 'enum' in col_type:
#                 enum_vals = col_type.split("'")[1::2]
#                 df_ver[col] = df_ver[col].fillna(enum_vals[0])
#             elif 'date' in col_type or 'time' in col_type:
#                 df_ver[col] = df_ver[col].fillna(pd.Timestamp('1970-01-01'))
#             else:
#                 df_ver[col] = df_ver[col].fillna('')

In [85]:
# # Ambil data mentah surattugas
# df_st_raw = pd.read_sql("SELECT idsurat, waktu FROM surattugas", db_old)

# # Cek tipe data
# print("Tipe data:", df_st_raw['waktu'].dtype)

# # Lihat nilai yang tidak bisa di-parse
# def coba_parse(val):
#     try:
#         pd.to_datetime(val)
#         return True
#     except:
#         return False

# mask_error = ~df_st_raw['waktu'].apply(coba_parse)
# print(f"\nBaris yang gagal di-parse: {mask_error.sum()}")
# display(df_st_raw[mask_error].head(10))

# # Tampilkan juga yang null
# print(f"\nJumlah NULL: {df_st_raw['waktu'].isnull().sum()}")
# display(df_st_raw[df_st_raw['waktu'].isnull()].head(5))

In [86]:
# # Ambil data dari DB lama
# df_sk_raw = pd.read_sql("SELECT * FROM suratkeluar", db_old)

# # Mapping & bersihin
# df_sk = pd.DataFrame({
#     'keterangan_sk': df_sk_raw['keterangan'].fillna(''),
#     'link_dokumen_sk': df_sk_raw['link'].fillna(''),
#     'status_sk': df_sk_raw['status'],   # nanti mapping enum kalau perlu
#     'nomor_sk': df_sk_raw['nosurat'].fillna(''),   # sekarang varchar(255) aman
#     'catatan_sk': df_sk_raw['catatan'].fillna(''),
#     'created_at': pd.to_datetime(df_sk_raw['created_at']),
#     'id_user': df_sk_raw['idusers'].fillna('')   # pastikan id_user ada di tabel users
# })
# # id_sk tidak dimasukkan (auto_increment)

## 3. Transform Data (jika diperlukan)

In [87]:
import pickle

# Gabungkan semua DataFrame final yang sudah jadi
fase3_data = {
    "sop_kategori": df_sop_kategori,
    "sop": df_sop,
    "surat_keluar": df_sk,
    "verifikasi_surat_keluar": df_verif_final,
    "surat_tugas": df_st,
    "surat_tugas_anggota": df_sta,   # tanpa id_sop
}

# Simpan ke file pickle
with open("fase_3_afrida.pkl", "wb") as f:
    pickle.dump(fase3_data, f)

print("✅ fase_3_afrida.pkl berhasil disimpan!")
print("📦 Isi:")
for nama, df in fase3_data.items():
    print(f"   - {nama}: {len(df)} baris, kolom: {list(df.columns)}")

✅ fase_3_afrida.pkl berhasil disimpan!
📦 Isi:
   - sop_kategori: 3 baris, kolom: ['id_sop_kategori', 'nama_kategori_sop']
   - sop: 4 baris, kolom: ['judul_sop', 'link_dokumen_sop', 'created_at', 'id_sop_kategori']
   - surat_keluar: 213 baris, kolom: ['id_sk', 'id_user', 'keterangan_sk', 'link_dokumen_sk', 'status_sk', 'nomor_sk', 'catatan_sk', 'created_at']
   - verifikasi_surat_keluar: 477 baris, kolom: ['id_sk', 'status_verifikasi_sk', 'catatan_verifikasi_sk', 'created_at']
   - surat_tugas: 135 baris, kolom: ['id_st', 'id_user', 'acara', 'undangan', 'waktu_acara', 'lokasi_acara', 'jenis_kegiatan', 'status_st', 'nomor_st', 'catatan_st', 'link_st', 'link_laporan', 'catatan_laporan', 'keterangan_st', 'catatan_pembatalan', 'created_at']
   - surat_tugas_anggota: 304 baris, kolom: ['id_st', 'id_user']


In [88]:
display(df_sk.head())

,id_sk,id_user,keterangan_sk,link_dokumen_sk,status_sk,nomor_sk,catatan_sk,created_at
0,24,U00011,Surat Permohonan Uji Kompetensi dan Penggunaan...,https://docs.google.com/document/d/1xyhc4T-FlR...,Sudah Revisi,090A/LEAP/IX/2023- 090B/LEAP/IX/2023,Lengkapi Tanggal Pelaksanaan dengan tanggal ya...,2023-09-11 17:57:49
1,26,U00011,Beasiswa Siswa GE,https://docs.google.com/document/d/1jMTK_3b57e...,Disetujui,101/LEAP/BD/X/2023,,2023-10-25 16:57:38
2,27,U00011,PERJANJIAN KERJASAMA / MEMORANDUM OF UNDERSTAN...,https://docs.google.com/document/d/1Edeb7hWaBq...,Disetujui,102/LEAP/BD/X/2023,,2023-10-26 17:37:59
3,28,U00026,Sertifikat / Sertifikat Kelas APK Private / 1 ...,https://drive.google.com/drive/folders/1JMrPJF...,Disetujui,Sertif 002a/APEX/XI/2324/02 (Page 1) dan 002b/...,,2023-11-10 16:10:04
4,29,U00011,Penawaran Pelatihan Business English ke PT. La...,https://docs.google.com/document/d/1wBK62noEJw...,Disetujui,105/LEAP/BD/XI/2023,,2023-11-13 15:41:58


## 4. Insert ke DB Baru

## 5. Verifikasi Data

## 6. Return Hasil Migrasi untuk migrate_db.py

## 7. Close Connection